In [1]:
import pandas as pd
import geopandas as gpd

# 1. grid_master_final 불러오기 (인구+시설 통합본)
grid_master = gpd.read_file('grid_master_final.shp')
print("grid_master 격자 수:", len(grid_master))
print(grid_master.columns.tolist())

# 2. 03번 도로망 불러오기
road_gdf = gpd.read_file('03._상세도로망_lev6.geojson')
print("\n도로망 링크 수:", len(road_gdf))
print(road_gdf.columns.tolist())
print(road_gdf.crs)

grid_master 격자 수: 11388
['grid_id', 'm_10g_pop', 'm_20g_pop', 'm_30g_pop', 'm_40g_pop', 'm_50g_pop', 'm_60g_pop', 'w_10g_pop', 'w_20g_pop', 'w_30g_pop', 'w_40g_pop', 'w_50g_pop', 'w_60g_pop', 'work_00h', 'work_01h', 'work_02h', 'work_03h', 'work_04h', 'work_05h', 'work_06h', 'work_07h', 'work_08h', 'work_09h', 'work_10h', 'work_11h', 'work_12h', 'work_13h', 'work_14h', 'work_15h', 'work_16h', 'work_17h', 'work_18h', 'work_19h', 'work_20h', 'work_21h', 'work_22h', 'work_23h', 'visi_00h', 'visi_01h', 'visi_02h', 'visi_03h', 'visi_04h', 'visi_05h', 'visi_06h', 'visi_07h', 'visi_08h', 'visi_09h', 'visi_10h', 'visi_11h', 'visi_12h', 'visi_13h', 'visi_14h', 'visi_15h', 'visi_16h', 'visi_17h', 'visi_18h', 'visi_19h', 'visi_20h', 'visi_21h', 'visi_22h', 'visi_23h', 'wd_w_pop', 'wd_v_pop', 'we_w_pop', 'we_v_pop', 'pop_peak_h', 'pop_elderl', 'pop_lunch_', 'park_cnt', 'bus_cnt', 'subway_cnt', 'shade_cnt', 'senior_cen', 'elder_clas', 'elder_welf', 'cooling_sh', 'daycare_cn', 'teen_fac_c', 'smart_s

In [5]:
print("grid_master CRS:", grid_master.crs)

grid_master CRS: EPSG:4326


In [3]:
# pop_peak_h(낮 시간대 방문인구 피크) 상위 격자 확인
top_flow = grid_master.nlargest(20, 'pop_peak_h')[['grid_id', 'pop_peak_h', 'wd_v_pop', 'we_v_pop', 'geometry']]
print(top_flow)

       grid_id  pop_peak_h   wd_v_pop   we_v_pop  \
476      23238   18.609609  10.182697  16.067825   
3729     32636   14.586708   9.289615   6.677373   
529      23412   14.541299   7.662993  11.835259   
707      23896   13.180288   9.023275   9.296202   
10062    50277   13.105663   8.614087  10.935921   
10063    50278   12.607966   7.805608  10.404373   
10061    50275   12.332186   8.151138   9.176020   
636      23681   12.314503   8.085732   8.229317   
5700     38207   12.162717   7.931177   7.128746   
582      23501   12.161099   7.667976   8.142900   
581      23499   12.117423   7.715465   7.429272   
3642     32439   12.095628   6.949684   6.563851   
3730     32638   12.005204   7.493491   5.057803   
477      23240   11.794780   5.266171  12.458848   
9971     50062   11.739516   4.123620   8.437933   
580      23498   11.250048   6.811917   7.249040   
5613     37968   11.013949   6.648370   6.770306   
5754     38320   10.984870   6.082227   7.781635   
3991     334

In [4]:
import folium

# 중심 좌표(격자들 평균 위치)로 지도 생성
center_lat = top_flow.geometry.centroid.y.mean()
center_lon = top_flow.geometry.centroid.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=15)

# 1. 전체 격자(grid_master) 경계선 - 얇은 회색 선으로
folium.GeoJson(
    grid_master.geometry,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': 'gray',
        'weight': 0.5,
        'fillOpacity': 0
    }
).add_to(m)

# 2. 유동량 상위 20개 격자는 빨간 테두리로 강조
folium.GeoJson(
    top_flow.geometry,
    style_function=lambda x: {
        'fillColor': 'red',
        'color': 'red',
        'weight': 1.5,
        'fillOpacity': 0.4
    }
).add_to(m)

# 3. 각 상위 격자 중심에 마커(팝업)도 같이
for idx, row in top_flow.iterrows():
    centroid = row.geometry.centroid
    folium.CircleMarker(
        location=[centroid.y, centroid.x],
        radius=4,
        popup=f"grid_id: {row['grid_id']}<br>pop_peak_h: {row['pop_peak_h']:.2f}",
        color='darkred',
        fill=True,
        fill_color='darkred'
    ).add_to(m)

m

/tmp/ipykernel_9/883018900.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = top_flow.geometry.centroid.y.mean()
/tmp/ipykernel_9/883018900.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = top_flow.geometry.centroid.x.mean()


In [7]:
# 유동량 상위 20개 격자에 subway_cnt, bus_cnt 같이 확인
top_flow_with_transit = grid_master.nlargest(20, 'pop_peak_h')[
    ['grid_id', 'pop_peak_h', 'subway_cnt', 'bus_cnt', 'geometry']
]
print(top_flow_with_transit)

       grid_id  pop_peak_h  subway_cnt  bus_cnt  \
476      23238   18.609609           0        1   
3729     32636   14.586708           0        0   
529      23412   14.541299           0        1   
707      23896   13.180288           0        0   
10062    50277   13.105663           0        0   
10063    50278   12.607966           0        0   
10061    50275   12.332186           0        0   
636      23681   12.314503           0        0   
5700     38207   12.162717           0        0   
582      23501   12.161099           0        0   
581      23499   12.117423           0        0   
3642     32439   12.095628           0        0   
3730     32638   12.005204           0        0   
477      23240   11.794780           0        0   
9971     50062   11.739516           0        0   
580      23498   11.250048           0        0   
5613     37968   11.013949           0        0   
5754     38320   10.984870           0        1   
3991     33424   10.968557     

In [8]:
# 유동량 상위 20개 격자에 고령인구(pop_elderl), 노인시설 컬럼까지 같이 확인
top_flow_full = grid_master.nlargest(20, 'pop_peak_h')[
    ['grid_id', 'pop_peak_h', 'subway_cnt', 'bus_cnt', 
     'pop_elderl', 'senior_cen', 'elder_clas', 'elder_welf', 'geometry']
]
print(top_flow_full)

       grid_id  pop_peak_h  subway_cnt  bus_cnt  pop_elderl  senior_cen  \
476      23238   18.609609           0        1   3429.4725           0   
3729     32636   14.586708           0        0   1030.2575           0   
529      23412   14.541299           0        1   3056.2625           0   
707      23896   13.180288           0        0   1462.0750           0   
10062    50277   13.105663           0        0   3608.9950           0   
10063    50278   12.607966           0        0   3237.8375           0   
10061    50275   12.332186           0        0   2600.4050           0   
636      23681   12.314503           0        0   1414.0325           0   
5700     38207   12.162717           0        0   1888.8875           0   
582      23501   12.161099           0        0   2264.4575           0   
581      23499   12.117423           0        0   1493.4900           0   
3642     32439   12.095628           0        0   1506.8075           0   
3730     32638   12.00520

In [9]:
# 고령인구 유동인구(pop_elderl) 상위 20개 격자 별도 확인
top_elderly = grid_master.nlargest(20, 'pop_elderl')[
    ['grid_id', 'pop_elderl', 'pop_peak_h', 'senior_cen', 'elder_clas', 'elder_welf', 'geometry']
]
print(top_elderly)

       grid_id  pop_elderl  pop_peak_h  senior_cen  elder_clas  elder_welf  \
10062    50277   3608.9950   13.105663           0           0           0   
476      23238   3429.4725   18.609609           0           0           0   
10063    50278   3237.8375   12.607966           0           0           0   
529      23412   3056.2625   14.541299           0           0           0   
10061    50275   2600.4050   12.332186           0           0           0   
582      23501   2264.4575   12.161099           0           0           0   
10100    50404   2237.4550    9.756095           0           0           0   
10101    50405   2160.7300   10.024334           0           0           0   
6445     40242   2112.5825    9.423273           0           0           0   
5700     38207   1888.8875   12.162717           0           0           0   
530      23414   1720.4500   10.481479           0           0           0   
6400     40134   1682.5825    8.619102           0           0  

In [10]:
# 핵심 후보 격자들의 시설 현황(bus_cnt, park_cnt 등) 종합 확인
core_candidates = grid_master[grid_master['grid_id'].isin(
    [23238, 50277, 23412, 50278, 50275, 23501, 38207, 23240, 38320]
)][['grid_id', 'pop_peak_h', 'pop_elderl', 'bus_cnt', 'park_cnt', 
    'work_08h', 'work_18h', 'wd_v_pop', 'we_v_pop', 'geometry']]
print(core_candidates)

       grid_id  pop_peak_h  pop_elderl  bus_cnt  park_cnt  work_08h  work_18h  \
476      23238   18.609609   3429.4725        1         0  5.867813  5.980014   
477      23240   11.794780   1569.2275        0         0  4.963398  4.356259   
529      23412   14.541299   3056.2625        1         0  4.999922  4.523233   
582      23501   12.161099   2264.4575        0         0  3.181172  3.602041   
5700     38207   12.162717   1888.8875        0         0  2.755348  3.739203   
5754     38320   10.984870   1562.2125        1         0  3.316271  4.241928   
10061    50275   12.332186   2600.4050        0         0  3.204570  3.061228   
10062    50277   13.105663   3608.9950        0         0  3.328281  3.283011   
10063    50278   12.607966   3237.8375        0         0  2.944888  3.030125   

        wd_v_pop   we_v_pop                                           geometry  
476    10.182697  16.067825  POLYGON ((127.12872 37.41185, 127.12838 37.411...  
477     5.266171  12.458848

In [11]:
import folium

# 9개 핵심 후보 격자
core_ids = [23238, 23240, 23412, 23501, 38207, 38320, 50275, 50277, 50278]
core_candidates = grid_master[grid_master['grid_id'].isin(core_ids)]

center_lat = core_candidates.geometry.centroid.y.mean()
center_lon = core_candidates.geometry.centroid.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

for idx, row in core_candidates.iterrows():
    centroid = row.geometry.centroid
    folium.CircleMarker(
        location=[centroid.y, centroid.x],
        radius=7,
        popup=f"grid_id: {row['grid_id']}<br>pop_peak_h: {row['pop_peak_h']:.1f}<br>pop_elderl: {row['pop_elderl']:.0f}",
        color='blue',
        fill=True,
        fill_color='blue'
    ).add_to(m)

m

/tmp/ipykernel_9/4178315468.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = core_candidates.geometry.centroid.y.mean()
/tmp/ipykernel_9/4178315468.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = core_candidates.geometry.centroid.x.mean()


In [12]:
# 경로당/노인시설이 있는 격자 찾기
senior_facility_grids = grid_master[
    (grid_master['senior_cen'] > 0) | 
    (grid_master['elder_clas'] > 0) | 
    (grid_master['elder_welf'] > 0)
][['grid_id', 'senior_cen', 'elder_clas', 'elder_welf', 'pop_elderl', 'geometry']]

print(f"노인시설 있는 격자 수: {len(senior_facility_grids)}")
print(senior_facility_grids.sort_values('pop_elderl', ascending=False).head(10))

노인시설 있는 격자 수: 71
       grid_id  senior_cen  elder_clas  elder_welf  pop_elderl  \
7703     43762           1           0           0     98.6700   
486      23255           0           1           0     64.6950   
5636     38006           1           0           0     59.1125   
3663     32474           1           0           0     57.4475   
10342    51069           1           0           0     51.3100   
4157     33882           1           0           0     43.7250   
7291     42608           0           1           0     43.4050   
7130     42164           1           0           0     40.2075   
489      23260           1           0           0     36.8875   
4670     35289           1           0           0     36.8525   

                                                geometry  
7703   POLYGON ((127.14124 37.37122, 127.14091 37.371...  
486    POLYGON ((127.13448 37.41194, 127.13414 37.411...  
5636   POLYGON ((127.07248 37.38418, 127.07214 37.384...  
3663   POLYGON ((127

In [13]:
from shapely.geometry import Point

# 아까 확인한 3개 핫스팟 중심 좌표
hotspots = {
    '야탑': (127.12873, 37.41131),   # 23412 대표
    '서현': (127.12397, 37.38474),   # 38207 대표
    '미금': (127.10891, 37.35016),   # 50277 대표
}

# 노인시설 격자 중심점 계산
senior_facility_grids['centroid'] = senior_facility_grids.geometry.centroid

# 각 핫스팟과의 거리 계산 (단위: 도 -> 대략 km 환산은 *111)
for name, (lon, lat) in hotspots.items():
    hp = Point(lon, lat)
    senior_facility_grids[f'dist_to_{name}_km'] = senior_facility_grids['centroid'].apply(
        lambda p: p.distance(hp) * 111
    )

# 야탑에서 가장 가까운 노인시설 격자 5개
print("=== 야탑에서 가까운 노인시설 ===")
print(senior_facility_grids.nsmallest(5, 'dist_to_야탑_km')[['grid_id','pop_elderl','dist_to_야탑_km']])

print("\n=== 서현에서 가까운 노인시설 ===")
print(senior_facility_grids.nsmallest(5, 'dist_to_서현_km')[['grid_id','pop_elderl','dist_to_서현_km']])

print("\n=== 미금에서 가까운 노인시설 ===")
print(senior_facility_grids.nsmallest(5, 'dist_to_미금_km')[['grid_id','pop_elderl','dist_to_미금_km']])

=== 야탑에서 가까운 노인시설 ===
      grid_id  pop_elderl  dist_to_야탑_km
978     24639      9.3475       0.447079
95      22214     10.8550       0.482040
930     24528      6.9100       0.552054
1260    25459     12.0150       0.589195
486     23255     64.6950       0.624342

=== 서현에서 가까운 노인시설 ===
      grid_id  pop_elderl  dist_to_서현_km
6212    39630      2.3475       0.592048
4636    35215      5.2975       0.673929
4490    34823     14.3325       0.675183
5989    38979     11.2025       0.697329
4283    34242     13.4500       0.766327

=== 미금에서 가까운 노인시설 ===
       grid_id  pop_elderl  dist_to_미금_km
10297    50926     14.9250       0.353932
10376    51198     10.0225       0.437732
9764     49478      4.1725       0.453011
10342    51069     51.3100       0.526856
10553    51645     20.3150       0.622648


/tmp/ipykernel_9/947873977.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  senior_facility_grids['centroid'] = senior_facility_grids.geometry.centroid


In [14]:
# 어린이집(daycare_cn) 있는 격자 확인
daycare_grids = grid_master[grid_master['daycare_cn'] > 0][
    ['grid_id', 'daycare_cn', 'pop_elderl', 'geometry']
]
print(f"어린이집 있는 격자 수: {len(daycare_grids)}")
print(daycare_grids.head(10))

어린이집 있는 격자 수: 76
     grid_id  daycare_cn  pop_elderl  \
100    22222           1      4.9175   
129    22269           1     35.9400   
130    22271           3     11.3275   
141    22289           1      6.2175   
142    22291           1      2.7375   
294    22766           1      5.1125   
363    22940           1     12.0200   
605    23624           1      2.0225   
882    24368           1     50.1250   
930    24528           1      6.9100   

                                              geometry  
100  POLYGON ((127.13269 37.41543, 127.13236 37.415...  
129  POLYGON ((127.12525 37.41504, 127.12491 37.415...  
130  POLYGON ((127.12592 37.41505, 127.12559 37.415...  
141  POLYGON ((127.13202 37.41515, 127.13168 37.415...  
142  POLYGON ((127.13270 37.41516, 127.13236 37.415...  
294  POLYGON ((127.13681 37.41360, 127.13647 37.413...  
363  POLYGON ((127.14190 37.41314, 127.14156 37.413...  
605  POLYGON ((127.13992 37.41094, 127.13958 37.410...  
882  POLYGON ((127.12269 37.4

In [16]:
# 낮시간대 직장인구 평균 (12~16시)
work_hours = ['work_12h', 'work_13h', 'work_14h', 'work_15h', 'work_16h']
grid_master['work_peak_daytime'] = grid_master[work_hours].mean(axis=1)

top_work_daytime = grid_master.nlargest(20, 'work_peak_daytime')[
    ['grid_id', 'work_peak_daytime', 'pop_peak_h', 'geometry']
]
print(top_work_daytime)

      grid_id  work_peak_daytime  pop_peak_h  \
3075    30721          12.683203    3.339897   
2128    27935          11.864856    3.260252   
2195    28060          11.035673    3.004710   
3642    32439          10.374519   12.095628   
2191    28053          10.122189    2.058629   
2192    28055           9.247894    2.452815   
2852    29978           9.137760    6.157085   
476     23238           9.015547   18.609609   
3643    32441           8.795000   10.240992   
8417    45719           8.322930    5.949586   
3730    32638           8.296192   12.005204   
3729    32636           8.138913   14.586708   
2794    29839           7.993114    7.530435   
2190    28052           7.728237    1.536053   
2058    27706           7.657769    1.589813   
529     23412           7.603257   14.541299   
2935    30270           7.573402    4.650261   
2341    28529           7.430589    3.387210   
2130    27939           7.322716    1.570656   
8486    45930           7.198695    4.60

In [17]:
import folium

work_top_ids = [30721, 27935, 28060, 32439, 28053, 28055, 29978, 32441, 45719, 32638, 32636, 29839, 27706, 30270, 28529, 27939, 45930]
work_top_candidates = grid_master[grid_master['grid_id'].isin(work_top_ids)]

center_lat = work_top_candidates.geometry.centroid.y.mean()
center_lon = work_top_candidates.geometry.centroid.x.mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

for idx, row in work_top_candidates.iterrows():
    centroid = row.geometry.centroid
    folium.CircleMarker(
        location=[centroid.y, centroid.x],
        radius=6,
        popup=f"grid_id: {row['grid_id']}<br>work_peak_daytime: {row['work_peak_daytime']:.1f}",
        color='green',
        fill=True,
        fill_color='green'
    ).add_to(m)

m

/tmp/ipykernel_9/1233286042.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lat = work_top_candidates.geometry.centroid.y.mean()
/tmp/ipykernel_9/1233286042.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  center_lon = work_top_candidates.geometry.centroid.x.mean()


In [20]:
# 미터 단위 좌표계(EPSG:5186)로 변환 후 계산
grid_master_m = grid_master.to_crs(epsg=5186)
origin_candidates_m = origin_candidates.to_crs(epsg=5186)
origin_candidates_m['centroid'] = origin_candidates_m.geometry.centroid

# 목적지 좌표도 같은 방식으로 변환 필요 - GeoDataFrame으로 만들어서 변환
from shapely.geometry import Point
import geopandas as gpd

dest_gdf = gpd.GeoDataFrame(
    {'name': list(destinations.keys())},
    geometry=[Point(lon, lat) for lon, lat in destinations.values()],
    crs='EPSG:4326'
).to_crs(epsg=5186)

for i, name in enumerate(destinations.keys()):
    dp = dest_gdf.geometry.iloc[i]
    origin_candidates_m[f'dist_{name}_km'] = origin_candidates_m['centroid'].distance(dp) / 1000
    nearest = origin_candidates_m.nsmallest(3, f'dist_{name}_km')[
        ['grid_id', 'senior_cen', 'daycare_cn', f'dist_{name}_km']
    ]
    print(f"\n=== {name}에서 가까운 출발지 후보 (정확한 거리) ===")
    print(nearest)


=== 야탑에서 가까운 출발지 후보 (정확한 거리) ===
     grid_id  senior_cen  daycare_cn  dist_야탑_km
978    24639           1           0    0.405521
990    24659           0           1    0.467272
930    24528           1           1    0.471090

=== 서현에서 가까운 출발지 후보 (정확한 거리) ===
      grid_id  senior_cen  daycare_cn  dist_서현_km
5970    38947           0           1    0.490122
6212    39630           1           0    0.557200
5989    38979           1           0    0.571263

=== 미금에서 가까운 출발지 후보 (정확한 거리) ===
       grid_id  senior_cen  daycare_cn  dist_미금_km
10297    50926           1           0    0.329592
10376    51198           1           0    0.427103
9764     49478           1           0    0.437187

=== 판교역에서 가까운 출발지 후보 (정확한 거리) ===
      grid_id  senior_cen  daycare_cn  dist_판교역_km
3171    31042           0           1     0.142472
3021    30582           1           0     0.437277
3868    33040           1           0     0.532316

=== 정자역에서 가까운 출발지 후보 (정확한 거리) ===
      grid_id  senior_ce

In [22]:
routes_expanded = []

for name, (lon, lat) in destinations.items():
    dp = Point(lon, lat)
    dp_m = gpd.GeoSeries([dp], crs='EPSG:4326').to_crs(epsg=5186).iloc[0]
    
    origin_candidates_m[f'dist_{name}'] = origin_candidates_m['centroid'].distance(dp_m) / 1000
    
    # 500m~1.5km 범위만 필터링
    in_range = origin_candidates_m[
        (origin_candidates_m[f'dist_{name}'] >= 0.5) & 
        (origin_candidates_m[f'dist_{name}'] <= 1.5)
    ]
    
    for _, row in in_range.iterrows():
        routes_expanded.append({
            '목적지': name,
            '출발지_grid_id': row['grid_id'],
            '거리_km': round(row[f'dist_{name}'], 3),
            '시설유형': 'senior' if row['senior_cen']>0 else 'daycare'
        })

routes_df = pd.DataFrame(routes_expanded)
print(f"500m~1.5km 조건 만족하는 동선 후보: {len(routes_df)}개\n")
print(routes_df.sort_values(['목적지', '거리_km']))

500m~1.5km 조건 만족하는 동선 후보: 111개

    목적지  출발지_grid_id  거리_km     시설유형
61   미금        51645  0.608   senior
55   미금        49046  0.637   senior
54   미금        48980  0.666  daycare
57   미금        50366  0.735  daycare
56   미금        49792  0.778   senior
..  ...          ...    ...      ...
80  판교역        34823  1.362   senior
79  판교역        34242  1.379   senior
66  판교역        25417  1.394   senior
71  판교역        27219  1.395   senior
85  판교역        35681  1.462   senior

[111 rows x 4 columns]


In [23]:
routes_top3 = routes_df.sort_values('거리_km').groupby('목적지').head(3).reset_index(drop=True)

print(f"총 동선 후보: {len(routes_top3)}개\n")
print(routes_top3.sort_values(['목적지', '거리_km']))

총 동선 후보: 15개

    목적지  출발지_grid_id  거리_km     시설유형
8    미금        51645  0.608   senior
9    미금        49046  0.637   senior
11   미금        48980  0.666  daycare
5    서현        39630  0.557   senior
6    서현        38979  0.571   senior
7    서현        35772  0.597  daycare
0    야탑        23255  0.501  daycare
1    야탑        22271  0.504  daycare
2    야탑        22289  0.520  daycare
12  정자역        35914  0.722   senior
13  정자역        36684  0.746  daycare
14  정자역        32474  1.035   senior
3   판교역        33040  0.532   senior
4   판교역        31458  0.535  daycare
10  판교역        34981  0.666  daycare


In [24]:
import folium

# 목적지 좌표
destinations = {
    '야탑': (127.12873, 37.41131),
    '서현': (127.12397, 37.38474),
    '미금': (127.10891, 37.35016),
    '판교역': (127.10979, 37.39614),
    '정자역': (127.11121, 37.39373),
}

colors = {'야탑':'red', '서현':'blue', '미금':'green', '판교역':'purple', '정자역':'orange'}

center_lat = sum([lat for lon,lat in destinations.values()]) / 5
center_lon = sum([lon for lon,lat in destinations.values()]) / 5

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# 목적지 마커 (별 아이콘)
for name, (lon, lat) in destinations.items():
    folium.Marker(
        location=[lat, lon],
        popup=f"{name} (목적지)",
        icon=folium.Icon(color=colors[name], icon='star')
    ).add_to(m)

# 출발지 마커 + 목적지까지 직선
for _, row in routes_top3.iterrows():
    dest_name = row['목적지']
    origin_id = row['출발지_grid_id']
    dest_lon, dest_lat = destinations[dest_name]
    
    origin_row = grid_master[grid_master['grid_id'] == origin_id].iloc[0]
    origin_pt = origin_row.geometry.centroid
    
    icon_type = 'home' if row['시설유형']=='senior' else 'child'
    folium.Marker(
        location=[origin_pt.y, origin_pt.x],
        popup=f"{dest_name} 출발지 (grid_id: {origin_id}, {row['시설유형']}, {row['거리_km']}km)",
        icon=folium.Icon(color=colors[dest_name], icon=icon_type)
    ).add_to(m)
    
    folium.PolyLine(
        locations=[[origin_pt.y, origin_pt.x], [dest_lat, dest_lon]],
        color=colors[dest_name],
        weight=2,
        dash_array='5, 5'
    ).add_to(m)

m

In [25]:
from shapely.geometry import Point
import geopandas as gpd

p1 = gpd.GeoSeries([Point(127.10979, 37.39614)], crs='EPSG:4326').to_crs(epsg=5186)  # 판교역
p2 = gpd.GeoSeries([Point(127.11121, 37.39373)], crs='EPSG:4326').to_crs(epsg=5186)  # 정자역

dist_km = p1.distance(p2).iloc[0] / 1000
print(f"판교역-정자역 거리: {dist_km:.3f} km")

판교역-정자역 거리: 0.296 km


In [30]:
routes_expanded_v3 = []
# 논문정자역 제외한 4개 목적지만

for name, (lon, lat) in destinations_v2.items():
    dp = Point(lon, lat)
    dp_m = gpd.GeoSeries([dp], crs='EPSG:4326').to_crs(epsg=5186).iloc[0]
    
    origin_candidates_m[f'dist_{name}'] = origin_candidates_m['centroid'].distance(dp_m) / 1000
    
    # 800m~1.5km 범위로 필터
    in_range = origin_candidates_m[
        (origin_candidates_m[f'dist_{name}'] >= 0.8) & 
        (origin_candidates_m[f'dist_{name}'] <= 1.5)
    ]
    
    for _, row in in_range.iterrows():
        routes_expanded_v3.append({
            '목적지': name,
            '출발지_grid_id': row['grid_id'],
            '거리_km': round(row[f'dist_{name}'], 3),
            '시설유형': 'senior' if row['senior_cen']>0 else 'daycare'
        })

routes_df_v3 = pd.DataFrame(routes_expanded_v3)
print(f"800m~1.5km 조건 만족하는 동선 후보: {len(routes_df_v3)}개\n")
print(routes_df_v3.sort_values(['목적지', '거리_km']))

800m~1.5km 조건 만족하는 동선 후보: 59개

    목적지  출발지_grid_id  거리_km     시설유형
38   미금        51938  0.813   senior
36   미금        51014  0.828   senior
37   미금        51374  0.845  daycare
39   미금        53048  1.257   senior
35   미금        47986  1.371   senior
34   미금        47446  1.404  daycare
40   미금        53199  1.405   senior
15   서현        33857  0.855  daycare
23   서현        40146  0.856   senior
20   서현        39725  0.879   senior
21   서현        39727  0.928  daycare
24   서현        41087  0.978   senior
22   서현        39930  1.029  daycare
13   서현        32474  1.037   senior
25   서현        41460  1.096  daycare
17   서현        34986  1.119   senior
18   서현        35361  1.215  daycare
29   서현        42369  1.231   senior
16   서현        34981  1.245  daycare
11   서현        31291  1.248  daycare
27   서현        41625  1.253   senior
14   서현        33040  1.254   senior
30   서현        42523  1.340  daycare
28   서현        42012  1.361  daycare
12   서현        31458  1.379  daycare
26   서현

In [31]:
routes_top3_v3 = routes_df_v3.sort_values('거리_km').groupby('목적지').head(3).reset_index(drop=True)

print(f"최종 동선 후보: {len(routes_top3_v3)}개\n")
print(routes_top3_v3.sort_values(['목적지', '거리_km']))

최종 동선 후보: 12개

    목적지  출발지_grid_id  거리_km     시설유형
0    미금        51938  0.813   senior
1    미금        51014  0.828   senior
2    미금        51374  0.845  daycare
3    서현        33857  0.855  daycare
4    서현        40146  0.856   senior
6    서현        39725  0.879   senior
9    야탑        23624  0.976  daycare
10   야탑        27219  1.012   senior
11   야탑        25417  1.080   senior
5   판교역        27807  0.874  daycare
7   판교역        26758  0.918  daycare
8   판교역        36684  0.965  daycare


In [32]:
import folium

destinations_v2 = {
    '야탑': (127.12873, 37.41131),
    '서현': (127.12397, 37.38474),
    '미금': (127.10891, 37.35016),
    '판교역': (127.10979, 37.39614),
}

colors = {'야탑':'red', '서현':'blue', '미금':'green', '판교역':'purple'}

center_lat = sum([lat for lon,lat in destinations_v2.values()]) / 4
center_lon = sum([lon for lon,lat in destinations_v2.values()]) / 4

m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

for name, (lon, lat) in destinations_v2.items():
    folium.Marker(
        location=[lat, lon],
        popup=f"{name} (목적지)",
        icon=folium.Icon(color=colors[name], icon='star')
    ).add_to(m)

for _, row in routes_top3_v3.iterrows():
    dest_name = row['목적지']
    origin_id = row['출발지_grid_id']
    dest_lon, dest_lat = destinations_v2[dest_name]
    
    origin_row = grid_master[grid_master['grid_id'] == origin_id].iloc[0]
    origin_pt = origin_row.geometry.centroid
    
    icon_type = 'home' if row['시설유형']=='senior' else 'child'
    folium.Marker(
        location=[origin_pt.y, origin_pt.x],
        popup=f"{dest_name} 출발지 (grid_id: {origin_id}, {row['시설유형']}, {row['거리_km']}km)",
        icon=folium.Icon(color=colors[dest_name], icon=icon_type)
    ).add_to(m)
    
    folium.PolyLine(
        locations=[[origin_pt.y, origin_pt.x], [dest_lat, dest_lon]],
        color=colors[dest_name],
        weight=2,
        dash_array='5, 5'
    ).add_to(m)

m

In [33]:
# 노드 컬럼 확인 (03번 도로망에 up_f_node, up_t_node 등이 있었음)
print(road_gdf[['link_id', 'up_f_node', 'up_t_node', 'dw_f_node', 'dw_t_node', 'length', 'oneway']].head(10))

# 결측치 확인 - 노드 ID가 비어있으면 그래프 연결이 끊김
print("\nup_f_node 결측:", road_gdf['up_f_node'].isna().sum())
print("up_t_node 결측:", road_gdf['up_t_node'].isna().sum())
print("length 결측:", road_gdf['length'].isna().sum())
print("length 0인 링크 수:", (road_gdf['length']==0).sum())

     link_id  up_f_node  up_t_node  dw_f_node  dw_t_node  length  oneway
0  572552600     352511     354179     354179     352511   0.074       0
1  572552638     353121     354180     354180     353121   0.045       0
2  572552640     354180     354181     354181     354180   0.140       0
3  572552678     353120     354181     354181     353120   0.060       0
4  572552820     352920     354186     354186     352920   0.109       0
5  572552832     352686     354189     354189     352686   0.041       0
6  572552834     353006     354189     354189     353006   0.009       0
7  572552838     354102     354109     354109     354102   0.136       0
8  572553002     563702     563703     563703     563702   0.013       0
9  572553018     353116     563704     563704     353116   0.211       0

up_f_node 결측: 0
up_t_node 결측: 0
length 결측: 0
length 0인 링크 수: 0


In [34]:
import networkx as nx

G = nx.Graph()

for idx, row in road_gdf.iterrows():
    G.add_edge(
        row['up_f_node'], 
        row['up_t_node'], 
        weight=row['length'],
        link_id=row['link_id']
    )

print(f"노드 수: {G.number_of_nodes()}")
print(f"엣지(도로 구간) 수: {G.number_of_edges()}")

# 연결성 확인 - 전체가 하나로 이어져 있는지
components = list(nx.connected_components(G))
print(f"\n연결된 덩어리(component) 개수: {len(components)}")
print(f"가장 큰 덩어리의 노드 수: {len(max(components, key=len))}")

노드 수: 2740
엣지(도로 구간) 수: 3462

연결된 덩어리(component) 개수: 4
가장 큰 덩어리의 노드 수: 2729


In [35]:
for i, comp in enumerate(sorted(components, key=len, reverse=True)):
    print(f"덩어리 {i+1}: 노드 {len(comp)}개")
    if len(comp) < 20:  # 작은 덩어리는 내용까지 출력
        print(f"  노드 목록: {list(comp)}")

덩어리 1: 노드 2729개
덩어리 2: 노드 4개
  노드 목록: [645866, 645868, 645870, 645863]
덩어리 3: 노드 4개
  노드 목록: [645865, 645867, 645869, 645871]
덩어리 4: 노드 3개
  노드 목록: [334440, 333124, 334439]


In [36]:
# 가장 큰 연결 덩어리만 남기기
largest_cc = max(components, key=len)
G_main = G.subgraph(largest_cc).copy()

print(f"정리 후 노드 수: {G_main.number_of_nodes()}")
print(f"정리 후 엣지 수: {G_main.number_of_edges()}")

# 확인 - 우리 12개 O-D 후보 출발지/목적지 노드가 이 큰 덩어리 안에 있는지 체크는
# 다음 단계(출발지-목적지를 노드에 매칭)에서 확인 예정

정리 후 노드 수: 2729
정리 후 엣지 수: 3454


In [37]:
from shapely.geometry import Point

# 1. 도로망의 각 노드 좌표를 딕셔너리로 만들기 (link_id 기준 노드 좌표 추출)
node_coords = {}
for idx, row in road_gdf.iterrows():
    coords = list(row.geometry.coords)
    node_coords[row['up_f_node']] = Point(coords[0])
    node_coords[row['up_t_node']] = Point(coords[-1])

print(f"좌표 확보된 노드 수: {len(node_coords)}")

# 2. 격자 중심점에서 가장 가까운 도로 노드 찾는 함수
def find_nearest_node(grid_id, grid_master, node_coords, G_main):
    grid_row = grid_master[grid_master['grid_id'] == grid_id].iloc[0]
    grid_centroid = grid_row.geometry.centroid
    
    min_dist = float('inf')
    nearest_node = None
    for node_id, node_pt in node_coords.items():
        if node_id in G_main.nodes:  # 큰 덩어리 안에 있는 노드만
            dist = grid_centroid.distance(node_pt)
            if dist < min_dist:
                min_dist = dist
                nearest_node = node_id
    return nearest_node, min_dist

# 3. 야탑 목적지로 테스트 (grid_id 대신 좌표라 임시로 가장 가까운 격자 찾기)
test_grid_id = 23412  # 야탑 근처 격자
nearest_node, dist = find_nearest_node(test_grid_id, grid_master, node_coords, G_main)
print(f"\n격자 {test_grid_id} → 가장 가까운 노드: {nearest_node} (거리: {dist:.6f}도)")

좌표 확보된 노드 수: 2740

격자 23412 → 가장 가까운 노드: 353073 (거리: 0.000221도)


In [38]:
# 12개 동선 후보의 출발지 grid_id 목록
origin_ids = routes_top3_v3['출발지_grid_id'].tolist()

# 4개 목적지도 각각 대표 grid_id로 매칭 (아까 썼던 대표 격자)
dest_grid_ids = {
    '야탑': 23412,
    '서현': 38207,
    '미금': 50277,
    '판교역': 30721,
}

# 출발지 노드 매칭
origin_nodes = {}
for gid in origin_ids:
    node, dist = find_nearest_node(gid, grid_master, node_coords, G_main)
    origin_nodes[gid] = node
    print(f"출발지 grid_id {gid} → 노드 {node} (거리 {dist*111000:.0f}m)")

print()

# 목적지 노드 매칭
dest_nodes = {}
for name, gid in dest_grid_ids.items():
    node, dist = find_nearest_node(gid, grid_master, node_coords, G_main)
    dest_nodes[name] = node
    print(f"목적지 {name}(grid_id {gid}) → 노드 {node} (거리 {dist*111000:.0f}m)")

출발지 grid_id 51938 → 노드 335921 (거리 183m)
출발지 grid_id 51014 → 노드 337251 (거리 64m)
출발지 grid_id 51374 → 노드 336791 (거리 64m)
출발지 grid_id 33857 → 노드 353457 (거리 127m)
출발지 grid_id 40146 → 노드 336914 (거리 83m)
출발지 grid_id 27807 → 노드 337479 (거리 46m)
출발지 grid_id 39725 → 노드 353742 (거리 96m)
출발지 grid_id 26758 → 노드 337975 (거리 116m)
출발지 grid_id 36684 → 노드 337648 (거리 75m)
출발지 grid_id 23624 → 노드 353401 (거리 32m)
출발지 grid_id 27219 → 노드 562807 (거리 31m)
출발지 grid_id 25417 → 노드 336194 (거리 165m)

목적지 야탑(grid_id 23412) → 노드 353073 (거리 25m)
목적지 서현(grid_id 38207) → 노드 353941 (거리 21m)
목적지 미금(grid_id 50277) → 노드 336214 (거리 26m)
목적지 판교역(grid_id 30721) → 노드 337723 (거리 9m)


In [39]:
routes_result = []

for _, row in routes_top3_v3.iterrows():
    dest_name = row['목적지']
    origin_id = row['출발지_grid_id']
    
    origin_node = origin_nodes[origin_id]
    dest_node = dest_nodes[dest_name]
    
    try:
        path = nx.shortest_path(G_main, source=origin_node, target=dest_node, weight='weight')
        path_length = nx.shortest_path_length(G_main, source=origin_node, target=dest_node, weight='weight')
        
        routes_result.append({
            '목적지': dest_name,
            '출발지_grid_id': origin_id,
            '출발지_노드': origin_node,
            '목적지_노드': dest_node,
            '경로_길이_km': round(path_length, 3),
            '경로_노드수': len(path),
            '경로': path
        })
        print(f"✅ {dest_name} ← {origin_id}: {path_length:.3f}km, 노드 {len(path)}개")
    except nx.NetworkXNoPath:
        print(f"❌ {dest_name} ← {origin_id}: 경로 없음 (연결 안 됨)")
        routes_result.append({
            '목적지': dest_name,
            '출발지_grid_id': origin_id,
            '경로_길이_km': None
        })

routes_result_df = pd.DataFrame(routes_result)
print(f"\n총 {len(routes_result_df)}개 중 성공: {routes_result_df['경로_길이_km'].notna().sum()}개")

✅ 미금 ← 51938: 0.950km, 노드 10개
✅ 미금 ← 51014: 1.312km, 노드 12개
✅ 미금 ← 51374: 1.265km, 노드 11개
✅ 서현 ← 33857: 1.068km, 노드 9개
✅ 서현 ← 40146: 1.138km, 노드 14개
✅ 판교역 ← 27807: 1.342km, 노드 21개
✅ 서현 ← 39725: 1.665km, 노드 17개
✅ 판교역 ← 26758: 0.869km, 노드 15개
✅ 판교역 ← 36684: 1.019km, 노드 12개
✅ 야탑 ← 23624: 1.275km, 노드 15개
✅ 야탑 ← 27219: 1.380km, 노드 19개
✅ 야탑 ← 25417: 1.362km, 노드 17개

총 12개 중 성공: 12개


In [41]:
import folium

center_lat = sum([lat for lon,lat in destinations_v2.values()]) / 4
center_lon = sum([lon for lon,lat in destinations_v2.values()]) / 4

m2 = folium.Map(location=[center_lat, center_lon], zoom_start=13)

# 1. 배경에 도로망 전체를 얇은 회색선으로
folium.GeoJson(
    road_gdf.geometry,
    style_function=lambda x: {'color': 'gray', 'weight': 1, 'opacity': 0.4}
).add_to(m2)

# 2. 목적지 마커
colors = {'야탑':'red', '서현':'blue', '미금':'green', '판교역':'purple'}
for name, (lon, lat) in destinations_v2.items():
    folium.Marker(
        location=[lat, lon],
        popup=f"{name} (목적지)",
        icon=folium.Icon(color=colors[name], icon='star')
    ).add_to(m2)

# 3. 실제 경로(노드 좌표 이어서) 그리기
for _, row in routes_result_df.iterrows():
    if row['경로_길이_km'] is None:
        continue
    dest_name = row['목적지']
    path_nodes = row['경로']
    
    # 경로의 노드들을 좌표로 변환
    path_coords = [[node_coords[n].y, node_coords[n].x] for n in path_nodes if n in node_coords]
    
    folium.PolyLine(
        locations=path_coords,
        color=colors[dest_name],
        weight=3,
        opacity=0.8,
        popup=f"{dest_name} ← {row['출발지_grid_id']} ({row['경로_길이_km']}km)"
    ).add_to(m2)
    
    # 출발지 마커
    origin_row = grid_master[grid_master['grid_id'] == row['출발지_grid_id']].iloc[0]
    origin_pt = origin_row.geometry.centroid
    folium.Marker(
        location=[origin_pt.y, origin_pt.x],
        popup=f"{dest_name} 출발지 (grid_id: {row['출발지_grid_id']})",
        icon=folium.Icon(color=colors[dest_name], icon='home')
    ).add_to(m2)

m2